# 5.3 커널 트릭: 선형분리가 안 되는 데이터 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter05_3_kernel_trick.ipynb)

책 본문: [5.3 커널 트릭: 선형분리가 안 되는 데이터](https://smhanlab.com/book-ml/kor/ml1/chapter05/3.html)

이 노트북은 책 5.3절의 내용을 코드로 재현합니다: (1) 동심원을 3차원으로 "올려보내면" 원형 경계가 평면이 되는 장면과, 같은 2차원 데이터에서 **선형 커널 vs RBF 커널**의 결정 경계를 비교하고, (2) **쌍대 문제를 2x2 커널 행렬로 손수 풀어** 5.1절의 primal 해와 일치함을 확인하고, (3) RBF 커널의 **무한차원 전개를 수치로** 검증하고, (4) `gamma`가 결정 경계에 미치는 영향을 4가지 값으로 보고, (5) XOR 사분면에서 **다항 커널 d=1(선형, 실패) vs d=2(성공)**을 비교하고, (6) RBF 커널 행렬이 항상 **양의 준정부호**(머서 조건)인지 수치로 점검합니다. numpy/scikit-learn만 씁니다.


## 1. 고차원으로 옮기면 "평평해진다": 3차원 리프트와 동심원

커널 트릭의 직관을 가장 직접적으로 보여주는 장면이다. 동심원 데이터(바깥쪽 원 = 클래스 0, 안쪽 원 = 클래스 1)를 2차원 특징 공간에서 보면 어떤 직선으로도 나눌 수 없다. 그러나 \(z = x_1^2 + x_2^2\)이라는 **세 번째 특징**을 추가해 3차원으로 올리면, 두 원은 \(z\)-고도가 서로 다른 두 개의 수평 고도에 놓이므로 **수평 평면 하나로 잘린다** — 3차원에서는 "휘어진 원형 경계"가 "평평한 평면"이 된다. 이 평면을 2차원으로 투영하면 바로 RBF SVM이 그리는 **원형 결정 경계**다.

왼쪽: \(z = x_1^2+x_2^2\) 위로 올려보낸 두 원환체. 오른쪽: 같은 3D 점군 위에 분리 평면 \(z = 0.5\)를 그렸다.

> 이 3차원 특징 \((x_1, x_2, x_1^2+x_2^2)\)은 RBF 커널이 암묵적으로 쓰는 무한차원 \(\phi\) 공간의 **3차원 "샘플"**이다 — 섹션 4에서 그 \(\phi\)가 실제로 **모든 도수의 모든 단항식**이라는 전개를 수치로 확인한다.


> 참고: `make_circles`의 실제 라벨은 **클래스 0 = 바깥쪽 원(반경 ~1.0), 클래스 1 = 안쪽 원(반경 ~0.4)** 이다. 라벨 번호 자체는 무관하다 — 커널 SVM은 "두 원환체가 고도 z로 나뉜다"는 구조를 라벨과 무관하게 찾기 때문이다.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # Colab에서는 /tmp 로 바꾸면 됨

# --- 동심원: 바깥쪽 원(반경 1.0, 클래스 0) / 안쪽 원(반경 0.4, 클래스 1) ---
from sklearn.datasets import make_circles
X, y = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=0)
z = X[:, 0]**2 + X[:, 1]**2            # 3차원으로 "올려보내는" 세 번째 특징

fig = plt.figure(figsize=(12.5, 5.2))

# (좌) z = x1^2+x2^2 위로: 두 수평 원환체
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
th = np.linspace(0, 2*np.pi, 60)
for r, c, lab in [(1.0, "blue", "class 0 (outer, z~1.0)"),
                  (0.4, "red",  "class 1 (inner, z~0.16")]:
    ax1.plot(r*np.cos(th), r*np.sin(th), np.full_like(th, r*r), color=c, lw=1.6, label=lab)
# 분리 평면 z = 0.5
g = np.linspace(-1.2, 1.2, 6)
GG, HH = np.meshgrid(g, g)
ax1.plot_surface(GG, HH, np.full_like(GG, 0.5), color="orange", alpha=0.25)
ax1.set_xlabel("x1"); ax1.set_ylabel("x2"); ax1.set_zlabel("z = x1^2+x2^2")
ax1.set_title("lifted to 3D: circles become two flat rings", fontsize=10)
ax1.legend(loc="upper left", fontsize=8)
ax1.view_init(elev=18, azim=-60)

# (우) 실제 200개 점(z = x1^2+x2^2) + 분리 평면
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
for cl, c in [(0, "blue"), (1, "red")]:
    sel = y == cl
    ax2.scatter(X[sel, 0], X[sel, 1], z[sel], c=c, s=14, alpha=0.7,
                label="class %d (n=%d)" % (cl, sel.sum()))
ax2.plot_surface(GG, HH, np.full_like(GG, 0.5), color="orange", alpha=0.2)
ax2.set_xlabel("x1"); ax2.set_ylabel("x2"); ax2.set_zlabel("z = x1^2+x2^2")
ax2.set_title("separating plane z = 0.5 cuts the two rings", fontsize=10)
ax2.legend(loc="upper left", fontsize=8)
ax2.view_init(elev=18, azim=-60)

fig.tight_layout()
fig.savefig(IMG + "/ch05_3_circles_3d_lift.svg")
plt.show()
print("class 0 (outer) mean z =", round(z[y == 0].mean(), 3),
      "  class 1 (inner) mean z =", round(z[y == 1].mean(), 3))


class 0 (outer) mean z = 0.997   class 1 (inner) mean z = 0.178


## 2. 선형 커널 vs RBF 커널: 동심원 (본문 실습 코드)

`make_circles`가 만드는 데이터는 어떤 직선으로도 못 나눈다. 같은 데이터를 선형 커널과 RBF 커널로 분류해 **결정 경계**와 **서포트 벡터 개수**를 함께 본다 — 서포트 벡터가 많을수록 모델이 "기억"하는 점이 많다는 뜻이다.


In [2]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

linear_svm = SVC(kernel="linear").fit(X_train, y_train)
rbf_svm    = SVC(kernel="rbf", gamma=2.0).fit(X_train, y_train)

print("linear kernel acc:", round(linear_svm.score(X_test, y_test), 3))
print("RBF kernel acc:   ", round(rbf_svm.score(X_test, y_test), 3))
print("SV count: linear", len(linear_svm.support_), " RBF", len(rbf_svm.support_))

def plot_boundary(ax, clf, title, data=X, ydata=y):
    h = 0.05
    lims = [data[:, 0].min() - 0.5, data[:, 0].max() + 0.5,
            data[:, 1].min() - 0.5, data[:, 1].max() + 0.5]
    xx, yy = np.meshgrid(np.arange(lims[0], lims[1], h), np.arange(lims[2], lims[3], h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap="coolwarm")
    ax.scatter(data[ydata == 0, 0], data[ydata == 0, 1], c="blue", s=15, label="class 0 (outer)")
    ax.scatter(data[ydata == 1, 0], data[ydata == 1, 1], c="red",  s=15, label="class 1 (inner)")
    ax.scatter(X_train[clf.support_][:, 0], X_train[clf.support_][:, 1],
               s=45, facecolors="none", edgecolors="k", lw=1.2, label="support vector")
    ax.set_title(title, fontsize=10); ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
plot_boundary(axes[0], linear_svm, "linear kernel  (test acc %.2f)" % linear_svm.score(X_test, y_test))
plot_boundary(axes[1], rbf_svm, "RBF kernel, gamma=2.0  (test acc %.2f)" % rbf_svm.score(X_test, y_test))
fig.tight_layout()
fig.savefig(IMG + "/ch05_3_kernel_boundaries.svg")
plt.show()


linear kernel acc: 0.517
RBF kernel acc:    1.0
SV count: linear 137  RBF 31


## 3. 커널 트릭의 심장: 쌍대 문제를 2x2 커널 행렬로 손수 풀어보기

5.1~5.2절에서는 **primal**(\(J(w,b)=\tfrac12\|w\|^2 + C\sum_i \xi^{(i)}\))을 직접 풀었다. 그런데 SVM의 **쌍대(dual) 문제**는 데이터가 \(w\cdot x\)가 아니라 **\(x_i\cdot x_j\)(= 커널값)로만 등장**하는 형태로 재작성된다 — 커널 트릭이 적용되는 바로 그 지점이다.

5.1절의 대칭 6개 데이터에서 서포트 벡터는 \((3,3)\)과 \((-3,-3)\) 두 점뿐이다. 라그랑주 변수는 6개지만, KKT 조건에 따라 **서포트 벡터가 아닌 점의 알파는 0**이므로 살아남는 알파는 두 개(\(a_1, a_4\))뿐이다. 제약 \(\sum_i a_i y_i = 0\)은 \(a_1 = a_4 = a\)를 강제하고, 쌍대 목적함수는 **일변수 2차함수**로 준다:

\[f(a) = 2a - \tfrac12\, a^2\, q, \qquad q = y_1^2 K_{11} + y_4^2 K_{44} + 2 y_1 y_4 K_{14} = 72\]

(서포트 벡터 두 점의 2x2 커널 행렬 \(K = \begin{pmatrix} 18 & -18 \\ -18 & 18 \end{pmatrix}\)에서 계산.) 꼭짓점 \(a^* = 2/q = 1/36\)에서 \(w = \sum_i a_i y_i x_i\)로 복원하면, 5.1절 손 계산 해 \(w=(1/6, 1/6),\ b=0\)가 **숫자 하나까지** 나와야 한다 — primal과 dual은 같은 해를 주므로(강렬한 쌍대성, strong duality).


In [3]:
# ---- dual: 서포트 벡터 2점의 2x2 커널 행렬로 손수 풀기 ----
C = 1.0
X6 = np.array([[ 3,  3], [ 4,  3], [ 3,  4],
               [-3, -3], [-4, -3], [-3, -4]], float)
y6 = np.array([1, 1, 1, -1, -1, -1], float)

sv = [0, 3]                              # (3,3), (-3,-3)
Xs, ys = X6[sv], y6[sv]
K = Xs @ Xs.T                            # 2x2 커널(내적) 행렬
print("2x2 kernel matrix K =\n", K)

# f(a) = 2a - (1/2) a^2 q ,  q = ys K ys  (대칭 a1=a4=a 이면)
q = float(ys @ K @ ys)
a_star = 2.0 / q                         # f'(a) = 2 - q a = 0 의 해
f_star = 2*a_star - 0.5*a_star**2*q
print(f"q = {q}   a* = 2/q = {a_star:.6f}   f(a*) = {f_star:.6f}")

# 꼭짓점 검증: 좌우로 가면 f가 작아져야 한다
for eps in (-0.001, 0.0, +0.001):
    aa = a_star + eps
    print(f"  f({aa:.4f}) = {2*aa - 0.5*aa*aa*q:.6f}")

# KKT로 w, b 복원:  w = sum_i a_i y_i x_i,  b = y_i - sum_j a_j y_j K(x_j, x_i)
a_s = np.array([a_star, a_star])         # (a1, a4) — 대칭이므로 동일
w_d = (a_s * ys) @ Xs                    # w = sum_i a_i y_i x_i
b_d = ys[0] - (a_s * ys) @ K[0]          # 서포트 벡터 i=0에서 b 복원
print("dual    -> w =", np.round(w_d, 6), " b =", round(b_d, 6))
print("matches 5.1 hand solution:",
      np.allclose(w_d, [1/6, 1/6], atol=1e-9) and abs(b_d) < 1e-9)

# 복원한 (w,b)가 6개 제약 y(w.x+b) >= 1 을 모두 만족하는지 확인
f6 = y6 * (X6 @ w_d + b_d)
print("functional margins y(w.x+b) =", np.round(f6, 4), " (all >= 1)")
print("geometric margin 2/||w|| =", round(2/np.linalg.norm(w_d), 4), " (5.1절: 8.4853)")
print("dual optimum f(a*) =", f"{f_star:.6f}", " == primal J = 0.5||w||^2 =", f"{0.5*w_d@w_d:.6f}")


2x2 kernel matrix K =
 [[ 18. -18.]
 [-18.  18.]]
q = 72.0   a* = 2/q = 0.027778   f(a*) = 0.027778
  f(0.0268) = 0.027742
  f(0.0278) = 0.027778
  f(0.0288) = 0.027742
dual    -> w = [0.166667 0.166667]  b = 0.0
matches 5.1 hand solution: True
functional margins y(w.x+b) = [1.     1.1667 1.1667 1.     1.1667 1.1667]  (all >= 1)
geometric margin 2/||w|| = 8.4853  (5.1절: 8.4853)
dual optimum f(a*) = 0.027778  == primal J = 0.5||w||^2 = 0.027778


## 4. RBF 커널은 "무한차원": 전개를 수치로 검증

RBF 커널 \(K(x,x') = e^{-\gamma\|x-x'\|^2}\)를 내적으로 분해한다. \(\|x-x'\|^2 = \|x\|^2 + \|x'\|^2 - 2x\cdot x'\)로 넣고, \(e^{2\gamma\, x\cdot x'}\)를 테일러 전개하면:

\[K(x,x') = e^{-\gamma\|x\|^2}\, e^{-\gamma\|x'\|^2}\, \sum_{k=0}^{\infty} \frac{(2\gamma)^k (x\cdot x')^k}{k!}\]

각 \(k\)-도 항을 단항식 \(x^\alpha\)(\(|\alpha|=k\))로 쪼개면 \(\phi(x)\)는 **모든 도수의 모든 단항식**에 가우시안 인자를 둔 **무한**개의 성분을 가진다:

\[\phi_\alpha(x) = e^{-\gamma\|x\|^2}\, \frac{(2\gamma)^{|\alpha|/2}\, x^\alpha}{\sqrt{\alpha!}}\]

이 \(\phi\)의 내적이 정확히 커널값을 주는지, 그리고 도수 합 \(k \le N\)으로 절단해도 수렴하는지를 수치로 확인한다. 참고: \(\phi(x)\)는 무한차원 벡터지만 유한 절단만으로도 꽤 정확한 근사가 된다 — "무한차원"이 실전 계산의 비용이 아니라는 뜻이다(실제 SVM은 \(\phi\) 자체를 계산하지 않고 커널값만으로 연산한다 — 섹션 3의 쌍대 문제 구조가 그 이유).


In [4]:
import math
gamma = 1.0

def phi_terms(x, N):
    """도수 합 |alpha| <= N 인 모든 성분 phi_alpha(x)의 리스트.
    phi_alpha(x) = e^{-gamma||x||^2} (2gamma)^{|alpha|/2} x^alpha / sqrt(alpha!)
    (e^{2 gamma x.x'} 의 계수 2가 x, x' 쪽에 대칭 배분: (2gamma)^{|alpha|/2} 씩)"""
    x1, x2 = x
    out = []
    for a1 in range(N + 1):
        for a2 in range(N - a1 + 1):
            k = a1 + a2
            out.append(math.exp(-gamma * (x1*x1 + x2*x2))
                       * (2 * gamma) ** (k / 2) * x1 ** a1 * x2 ** a2
                       / (math.sqrt(math.factorial(a1))
                          * math.sqrt(math.factorial(a2))))
    return out

def rbf(x, z, g=gamma):
    return math.exp(-g * np.sum((np.asarray(x) - np.asarray(z)) ** 2))

x, z = (1.2, -0.7), (0.5, 1.1)
exact = rbf(x, z)
print(f"K(x,z) = {exact:.10f}")
for N in [0, 1, 2, 5, 10, 20, 40]:
    s = sum(a * b for a, b in zip(phi_terms(x, N), phi_terms(z, N)))
    print(f"도수 합 <= {N:2d}: 부분합 = {s:.10f}   (오차 {s - exact:+.2e})")

px = phi_terms(x, 40)
print(f"\n||phi(x)||^2 = {sum(t*t for t in px):.10f}   (K(x,x)=1 이므로 항상 1)")


K(x,z) = 0.0239928358
도수 합 <=  0: 부분합 = 0.0337086769   (오차 +9.72e-03)
도수 합 <=  1: 부분합 = 0.0222477268   (오차 -1.75e-03)
도수 합 <=  2: 부분합 = 0.0241960883   (오차 +2.03e-04)
도수 합 <=  5: 부분합 = 0.0239927669   (오차 -6.90e-08)
도수 합 <= 10: 부분합 = 0.0239928358   (오차 +5.73e-15)
도수 합 <= 20: 부분합 = 0.0239928358   (오차 -4.16e-17)
도수 합 <= 40: 부분합 = 0.0239928358   (오차 -3.47e-17)

||phi(x)||^2 = 1.0000000000   (K(x,x)=1 이므로 항상 1)


## 5. gamma의 효과 — "구불 복잡도" 손잡이

`gamma`는 RBF \(K(x,x') = e^{-\gamma\|x-x'\|^2}\)에서 각 데이터점 주변의 **가우시안 "봉우리" 폭의 역수**에 해당한다(`gamma = 1/(2\sigma^2)`). 데이터를 고정하고 `gamma`만 \(0.05, 0.3, 1.0, 30\)으로 바꾸면: 작으면 경계가 거의 직선에 가깝게 뭉뚱그려지고(과소적합), 크면 각 점마다 봉우리가 생겨 경계가 학습 데이터 하나하나를 감싸는 형태로 심하게 구불거린다(과적합).


In [5]:
gammas = [0.05, 0.3, 1.0, 30.0]
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, g in zip(axes.ravel(), gammas):
    clf = SVC(kernel="rbf", gamma=g, C=1.0).fit(X_train, y_train)
    acc_tr, acc_te = clf.score(X_train, y_train), clf.score(X_test, y_test)
    print("gamma=%5.2f : train acc %.3f   test acc %.3f   SV %d"
          % (g, acc_tr, acc_te, len(clf.support_)))
    plot_boundary(ax, clf, "gamma = %s   (test acc %.2f)" % (g, acc_te))
fig.suptitle("RBF gamma sweep on concentric circles", fontsize=12)
fig.tight_layout()
fig.savefig(IMG + "/ch05_3_gamma_sweep.svg")
plt.show()


gamma= 0.05 : train acc 0.514   test acc 0.467   SV 136
gamma= 0.30 : train acc 0.993   test acc 1.000   SV 74


gamma= 1.00 : train acc 1.000   test acc 1.000   SV 34
gamma=30.00 : train acc 1.000   test acc 1.000   SV 85


## 6. 다항 커널: XOR은 "도수 2"가 정확히 필요하다

**XOR**(사분면) 데이터: \(y = -\mathrm{sign}(x_1 x_2)\)로, 1·3사분면(−1)과 2·4사분면(+1)이 **사분면 단위로** 교차한다. 이 패턴을 선형분리 불가능하게 만드는 핵심 단항식은 \(x_1 x_2\) — **도수 2**다. 도수 \(d\) 다항 커널 \((x\cdot x' + 1)^d\)는 도수 \(d\) 이하의 **모든 단항식**을 포함하므로, d=1(선형)은 원점을 기준으로 교차하는 양 클래스를 직선 하나로 못 나누어 50%에 머무르고, d=2는 \(x_1 x_2\) 항을 포함해 거의 완벽해야 한다.

\((x\cdot x'+1)^2\)를 다항식으로 전개하면 \((x_1x_1'+x_2x_2')^2 + 2(x_1x_1'+x_2x_2') + 1\)로, **도수 2, 1, 0** 단항식이 전부 섞여 있다. 명시적 \(\phi_2(x) = (x_1^2,\, x_2^2,\, \sqrt2\, x_1x_2,\, \sqrt2\, x_1,\, \sqrt2\, x_2,\, 1)\)의 내적이 커널값과 **숫자 하나까지** 같은지 확인한다 — "커널 트릭은 전개를 생략할 뿐, 같은 내적을 계산한다"는 명제의 직접 검증이다.


In [6]:
def phi2(x):
    # (x.x'+1)^2 를 도수 2 이하 단항식으로 전개하면 6개의 성분이 필요하다:
    # x1^2, x2^2, 2*x1*x2 (-> sqrt2 계수), 2*x1, 2*x2 (-> sqrt2 계수), 1
    x1, x2 = x
    s2 = math.sqrt(2)
    return np.array([x1 ** 2, x2 ** 2, s2 * x1 * x2, s2 * x1, s2 * x2, 1.0])

# 커널값과 명시적 내적이 일치하는지 확인 (커널 트릭의 "같은 내적")
rng = np.random.default_rng(0)
Xs = rng.normal(size=(50, 2))
P = np.array([phi2(x) for x in Xs])
K_poly = ((Xs @ Xs.T + 1) ** 2)                    # (x.x' + 1)^2
K_exp  = P @ P.T                                   # phi2(x).phi2(x')
print("max |K_poly - K_explicit| =", np.max(np.abs(K_poly - K_exp)))
assert np.max(np.abs(K_poly - K_exp)) < 1e-10
print("일치: (x.x'+1)^2 는 정확히 phi2의 내적이다")

# ---- XOR 사분면 데이터: y = -sign(x1*x2), 4개 사분면 클러스터 ----
rng2 = np.random.default_rng(3)
n = 80
quads  = np.array([[ 1,  1], [ 1, -1], [-1,  1], [-1, -1]])
# y = -sign(x1*x2): (1,1)->-1, (1,-1)->+1, (-1,1)->+1, (-1,-1)->-1
labels = np.array([-1, 1, 1, -1])
Xq = np.vstack([rng2.normal(q * 3, 0.8, (n, 2)) for q in quads])
yq = np.repeat(labels, n)
perm = rng2.permutation(len(yq)); Xq, yq = Xq[perm], yq[perm]

# 선형(d=1) vs 다항 d=2
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, d in zip(axes, [1, 2]):
    if d == 1:
        clf = SVC(kernel="linear", C=1.0)
        lab = "linear (d=1)"
    else:
        clf = SVC(kernel="poly", degree=2, coef0=1, C=1.0)
        lab = "polynomial d=2"
    clf.fit(Xq, yq)
    acc = clf.score(Xq, yq)
    print("%-16s : acc %.3f   SV %d" % (lab, acc, len(clf.support_)))
    h = 0.08
    lims = [Xq[:, 0].min() - 1, Xq[:, 0].max() + 1,
            Xq[:, 1].min() - 1, Xq[:, 1].max() + 1]
    xx, yy = np.meshgrid(np.arange(lims[0], lims[1], h), np.arange(lims[2], lims[3], h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap="coolwarm")
    for lbl, c in [(1, "red"), (-1, "blue")]:
        ax.scatter(Xq[yq == lbl, 0], Xq[yq == lbl, 1], c=c, s=14)
    ax.set_title("%s  (acc %.2f)" % (lab, acc), fontsize=10)
    ax.set_aspect("equal"); ax.tick_params(labelsize=8)
fig.suptitle("XOR quadrants: linear vs polynomial kernel degree 2", fontsize=12)
fig.tight_layout()
fig.savefig(IMG + "/ch05_3_poly_xor.svg")
plt.show()


max |K_poly - K_explicit| = 1.0658141036401503e-14
일치: (x.x'+1)^2 는 정확히 phi2의 내적이다
linear (d=1)     : acc 0.738   SV 319
polynomial d=2   : acc 1.000   SV 18


## 7. 머서 조건 수치 점검: RBF 커널 행렬은 항상 PSD

커널 함수가 유효하려면 데이터 \(m\)개에 대한 커널 행렬 \(K_{ij} = K(x_i, x_j)\)가 항상 **양의 준정부호**(모든 고유값 \(\ge 0\))여야 한다 — **머서(Mercer) 조건**이다. RBF 커널 행렬의 고유값을 실제 계산해본다.

부록: \(\phi\)가 무한차원이어도 \(m\)개 데이터의 커널 행렬은 최대 **rank m**이다 — 커널 방법의 "실효 차원"은 데이터 개수로 제한된다.


In [7]:
Xr = np.random.default_rng(11).normal(size=(60, 4))
g = 0.5
K = np.exp(-g * np.sum((Xr[:, None, :] - Xr[None, :, :]) ** 2, axis=2))
eig = np.linalg.eigvalsh(K)
print("min eigenvalue =", f"{eig.min():.3e}")
print("top-5 eigenvalues:", np.round(eig[-5:][::-1], 3))
print("rank (eig > 1e-9) =", int((eig > 1e-9).sum()), " out of", len(eig))
assert eig.min() > -1e-9
print("PASS: RBF 커널 행렬은 양의 준정부호 (머서 조건 성립)")


min eigenvalue = 5.071e-03
top-5 eigenvalues: [10.992  4.64   4.558  3.693  3.371]
rank (eig > 1e-9) = 60  out of 60
PASS: RBF 커널 행렬은 양의 준정부호 (머서 조건 성립)


## 정리: 다음으로

- **5.3절 핵심**: 커널 SVM은 \(w^Tx\)를 \(\phi(x)\)의 내적으로, 그 내적은 커널 함수 \(K(x,x')\)로 **우회**해 계산한다 — 고차원 변환을 명시적으로 하지 않으면서 비선형 경계를 얻는다.
- **6장 (정규화와 모델 선택)**: 이번 절의 \(C\), `gamma` 튜닝은 6장의 **교차 검증**으로 체계화된다 — 검증셋 정확도로 \((C, \gamma)\)를 함께 그리드 서치하는 것이 표준 관행이다.
- **10장 (신경망)**: RBF의 "봉우리"와 CNN의 **커널 필터**는 수학적으로 다른 개념이지만, "원본 특징을 고차 항으로 변환해 선형 분류"한다는 큰 그림은 이어진다 — 10장에서 이 연결을 다시 만난다.
